In [1]:
import pandas as pd 
import numpy as np
import mlflow
import mlflow.sklearn
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split,GridSearchCV
from sklearn.metrics import mean_squared_error
from sklearn.datasets import load_diabetes

In [8]:
diabete = load_diabetes()
print(diabete.feature_names)

['age', 'sex', 'bmi', 'bp', 's1', 's2', 's3', 's4', 's5', 's6']


In [13]:
df = pd.DataFrame(data=diabete.data, columns=diabete.feature_names)
df['target'] = diabete.target
df.head()
df.shape

(442, 11)

In [14]:
from urllib.parse import urlparse
X=df.drop(columns=['target'])
y=df['target']

In [16]:
def hiperparameter_tuning(X_train,y_train,param_grid):
    rf=RandomForestRegressor()
    grid_search=GridSearchCV(estimator=rf,param_grid=param_grid,cv=3,n_jobs=-1,verbose=2,scoring='neg_mean_squared_error')
    grid_search.fit(X_train,y_train)
    return grid_search

In [20]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20)

from mlflow.models import infer_signature
signature = infer_signature(X_train, y_train)


param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [5, 10, None],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2]
}

with mlflow.start_run():
    grid_search=hiperparameter_tuning(X_train,y_train,param_grid)
    best_model=grid_search.best_estimator_
    y_pred=best_model.predict(X_test)
    mse=mean_squared_error(y_test,y_pred)
    
    mlflow.log_param("best_params",grid_search.best_params_['n_estimators'])
    mlflow.log_param("max_depth",grid_search.best_params_['max_depth'])
    mlflow.log_param("min_samples_split",grid_search.best_params_['min_samples_split'])
    mlflow.log_param("min_samples_leaf",grid_search.best_params_['min_samples_leaf'])
    mlflow.log_metric("mse",mse)

    mlflow.set_tracking_uri("http://127.0.0.1:5000")
    tracking_url_type_store = urlparse(mlflow.get_tracking_uri()).scheme

    if tracking_url_type_store != "file":
        mlflow.sklearn.log_model(best_model, "model", registered_model_name="RandomForestDiabetesModel")
    else:
        mlflow.sklearn.log_model(best_model, "model", signature=signature)
    
    print("Best hyperparameters:", grid_search.best_params_)
    print("Mean Squared Error:", mse)
    
    



Fitting 3 folds for each of 24 candidates, totalling 72 fits


2025/12/31 08:38:07 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
Successfully registered model 'RandomForestDiabetesModel'.
2025/12/31 08:38:15 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: RandomForestDiabetesModel, version 1
Created version '1' of model 'RandomForestDiabetesModel'.


Best hyperparameters: {'max_depth': 5, 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 100}
Mean Squared Error: 2794.2226722474193
🏃 View run delightful-perch-734 at: http://127.0.0.1:5000/#/experiments/0/runs/f42abb56d76a449fb9f393b8f04c9ccf
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/0
